# 模型忽略关键实体怎么办？注意力权重分配机制引导生成聚焦重点

## 场景痛点

大语言模型虽然强大，但在特定任务中仍可能出现“走神”现象。

例如，在生成摘要时遗漏核心人物名字，在问答系统中无法准确抽取关键日期，或在对话中忽略用户强调的重点。这些情况导致生成内容偏离主题，缺乏关键信息，甚至误导用户。

## 根本原因

模型在处理文本时，注意力机制未能对关键实体给予足够关注。Transformer 架构通过注意力权重对输入 token 进行加权聚合，从而决定输出内容。如果某些重要实体的注意力权重偏低，它们就很难在最终输出中体现出来。


### 方案一： 提示词

通过精心设计的 prompt，可以间接引导模型关注特定内容。例如，明确要求模型在回答中使用某些关键词，或以特定结构组织内容。这种方式简单有效，适用于大多数模型




In [1]:
prompt = """
请用自然流畅的语言，深入探讨一下人工智能和大模型的未来发展趋势，并结合医疗、自动驾驶、智能客服等具体行业，分析它们的潜在应用和挑战。
请在你的回答中，尽可能自然地穿插以下词汇：大模型、人工智能、医疗、自动驾驶、智能客服。
"""

### 方案二： 自然语言处理
[命名实体识别](https://www.modelscope.cn/models/iic/nlp_seqgpt-560m)

借助命名实体识别技术，从输入中提取关键实体，并将其插入 prompt 或用于干预模型生成逻辑。该方法自动化程度高，能动态识别关键信息，但依赖外部模块，推理链更复杂。


### 方案三：修改 Attention 层，最终生成词汇的概率分布（logits）

更直接、有效的方式是干预模型输出层的 logits。logits 是模型对词汇表中每个词的“打分”，在它进入 softmax 之前修改，可以精确提升或降低特定词汇的生成概率。该方法不依赖 prompt，也不需要重新训练模型，适用于推理阶段实时干预。

In [ ]:
import torch
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks

######
# 核心优化策略：
# 1. 干预时机不要过早：修改 Attention 层属于模型内部干预，Attention 的主要作用是让模型关注输入序列的不同部分，
#   直接增强特定 token 的 attention score 并不直接等同于“让模型多生成这个词”，反而可能破坏模型对上下文的理解，导致生成内容不连贯。
# 2. 干预目标确保精确：更直接、有效的方法是直接干预最终生成词汇的概率分布（logits），
#   在它进入 softmax 之前修改，可以精确地提升或降低特定词汇的生成概率。
# 3. 关键词 Tokenization 问题：代码中使用 tokenizer.convert_tokens_to_ids 获取词汇 ID，
#   但像 “大模型”、“自动驾驶” 等词通常会被 tokenizer 拆分为多个 token（如 "大"、"模型"），只会关注第一个 token，效果有限。
# 4. 状态管理复杂：通过 model.current_input_ids 传递输入信息给 hook 函数的方式不够健壮，
#   且在生成多个 token 的过程中，input_ids 会变化，逻辑会变得很复杂。

# ================== 1. 加载模型与 Tokenizer ==================
model_name = "Qwen/Qwen3-0.6B"

print("正在加载模型和Tokenizer...")
# 使用 pipeline 加载模型
chat_pipeline = pipeline(
    task=Tasks.chat,
    model=model_name,
    device_map="auto"
)

# 获取模型和 tokenizer
model = chat_pipeline.model
tokenizer = chat_pipeline.tokenizer
print("模型加载完成！")


# ================== 2. 定义关键词和获取 Token IDs ==================
# 定义关键词列表
keywords = ["大模型", "人工智能", "医疗", "自动驾驶", "智能客服"]
focus_token_ids = []

for keyword in keywords:
    token_ids = tokenizer.encode(keyword, add_special_tokens=False)
    if token_ids:
        focus_token_ids.extend(token_ids)

# 去重，并转换为 tensor
focus_token_ids = torch.tensor(list(set(focus_token_ids)), device=model.device)
print(f"需要关注的关键词 Token ID: {focus_token_ids}")

# ================== 3. 定义 Hook 函数（作用于 Logits） ==================
def modify_logits_hook(module, input, output):
    """
    这个 hook 函数在 lm_head 计算出 logits 后被调用。
    它会直接提升我们关注的关键词的 logits 值。
    """
    logits = output[0] if isinstance(output, tuple) else output
    last_token_logits = logits[:, -1, :]

    #在使用 focus_token_ids 之前，确保它和 logits 在同一个设备上。避免 RuntimeError 的关键。
    device_aware_focus_ids = focus_token_ids.to(last_token_logits.device)

    bias = 3.0
    # 使用aware_focus_ids 进行索引
    last_token_logits[:, device_aware_focus_ids] += bias

    return output

# ================== 4. 注册与移除 Hook ==================
hook_handle = None

def add_hook():
    global hook_handle
    # 将 hook 注册到模型的 lm_head (输出层)，这是最直接影响生成结果的地方
    if hasattr(model, "lm_head") and hook_handle is None:
        print("Hook 已注册到 lm_head")
        hook_handle = model.lm_head.register_forward_hook(modify_logits_hook)
    else:
        print("未能找到 lm_head 或 Hook 已存在。")

def remove_hook():
    global hook_handle
    if hook_handle:
        hook_handle.remove()
        hook_handle = None
        print("Hook 已移除")

## ================== 5. 定义生成函数 ==================
def generate_text(prompt):
    """
    使用 pipeline 生成文本。
    """
    messages = [{"role":"user", "content": prompt}]

    response = chat_pipeline(
        messages,
        max_new_tokens=512,
        do_sample=True,
        temperture=0.7,
        top_p=0.9,
        repetition_penalty=1.05,
        no_repeat_ngram_size=2
    )

    print(response)

    content = response['messafe']['content']

    if '</think>' in content:
        content = content.split('</think>', 1)[-1].strip()
    return content

# ================== 6. 定义关键词统计函数 ==================
def count_keywords(text, keywords):
    count = 0
    present_keywords = []
    for word in keywords:
        if word in text:
            count += text.count(word)
            present_keywords.append(word)
    return count, present_keywords


# ================== 7. 测试对比 ==================
test_prompt = """
请用自然流畅的语言，深入探讨一下人工智能和大模型的未来发展趋势，并结合医疗、自动驾驶、智能客服等具体行业，分析它们的潜在应用和挑战。

"""

print("\n" + "="*20 + " 1. 不使用 Hook 生成 " + "="*20)

remove_hook()
output_no_hook = generate_text(test_prompt)
print(f"\n【生成结果】:\n{output_no_hook}")
count1, present1 = count_keywords(output_no_hook, keywords)
print(f"\n【关键词统计】: 数量: {count1}, 出现的词: {present1}")

print("\n" + "="*20 + " 2. 使用 Hook 生成（强调关键词） " + "="*20)
add_hook()
output_with_hook = generate_text(test_prompt)
print(f"\n【生成结果】:\n{output_with_hook}")
count2, present2 = count_keywords(output_with_hook, keywords)
print(f"\n【关键词统计】: 数量: {count2}, 出现的词: {present2}")


remove_hook()

### 通过干预 Logits 引导模型聚焦关键信息

对于像 Qwen 或 Llama 这样的先进自回归模型（Decoder-only Models），它们依赖自注意力机制来理解上下文。简而言之，模型在生成每个新词时，会回顾此前的所有文本，并从中提取相关信息。我们的目标是在这个“回顾”过程中施加影响，让模型更关注我们指定的关键内容。

## 高级干预策略

### 强力干预（The Hard Boost）

最直接的方式是在目标 token 的 logits 上添加一个固定的正向偏置。这种方式干预效果明显，但可能影响文本的自然性，导致关键词重复出现。可通过 `no_repeat_ngram_size` 等参数缓解这一问题。

### 温和引导（The Gentle Nudge）

更精细的做法是采用加权融合策略，将原始 logits 与关键词偏置进行线性融合：

```
new_logits = original_logits * (1 - α) + entity_bias * α
```

其中，α 是一个介于 0 和 1 之间的融合因子，用于控制干预强度。α 越小，干预越温和，生成内容越自然。

### 动态衰减

还可根据生成阶段动态调整偏置值。例如，在生成初期给予较强干预，随后逐步减弱，使模型在后期拥有更多自由发挥的空间，从而在聚焦关键信息与保持多样性之间取得平衡。

## 局限性

- **过度聚焦风险**：可能导致生成内容变得狭隘、重复，缺乏创造性。
- **计算开销**：虽然单次干预开销较小，但在复杂场景中频繁干预会略微增加推理延迟。

## 与其他技术的协同

强制聚焦并非孤立手段，它可与多种主流技术结合，实现更优效果：

- **提示词工程（Prompt Engineering）**：先用高质量 Prompt 指明方向，再通过干预确保关键细节不丢失。
- **RAG（检索增强生成）**：RAG 负责从外部知识库中检索关键信息，而干预技术则确保这些信息在最终输出中得以体现。
- **LoRA / QLoRA 微调**：通过微调让模型掌握特定领域知识，再在推理时用干预技术引导模型聚焦具体任务。

## 总结

通过钩子（Hook）机制干预模型的 Logits 层，是一种强大、可解释的干预方式。它能够引导模型在生成过程中聚焦关键实体，提升输出的准确性与相关性。结合提示词工程、RAG 和微调技术，可以进一步增强干预效果，使其在实际应用中更加稳定、自然地服务于特定任务需求。
